In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from scipy.stats import scoreatpercentile
from tqdm import tqdm
import time
import warnings


In [ ]:
# 1. 파일 이름 정의
SMC_INPUT_PATH = str(PROCESSED_DIR / 'smc_tidy.csv')

smc_df = pd.read_csv(SMC_INPUT_PATH, dtype='str')
smc_df


In [ ]:
columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float


In [ ]:
dtype_int = {col: target_int for col in columns_to_int}
smc_df = smc_df.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
smc_df = smc_df.astype(dtype_float)


In [ ]:
# 사용할 컬럼 목록 정의 (20개)
MODEL_COLUMNS = [
    'age', 'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_COLUMN = 'use_quan'
feature_columns = [c for c in MODEL_COLUMNS if c != TARGET_COLUMN]

smc_df = smc_df[MODEL_COLUMNS]

print("\n======= [INFO] 데이터 로드 및 전처리 완료 =======\n")


In [ ]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 0. 설정
# -------------------------------------------------------------
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'
GROUP_COLUMN = 'procedure_cd_fin' # 그룹핑 기준 변수

smc_df = smc_df.copy()
# -------------------------------------------------------------
# 1. 수술별(procedure_cd_fin) interquartile_range 및 임계값 계산
# -------------------------------------------------------------
print("[INFO] 수술별 그룹핑 및 IQR 계산 중...")

# (1) 각 수술 코드별로 lower_quartile, upper_quartile 계산하여 원본 데이터프레임에 임시 컬럼으로 붙임 (.transform 사용)
# transform은 그룹별로 계산한 값을 원래 행의 위치에 맵핑해줍니다.
smc_df['group_Q1'] = smc_df.groupby(GROUP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.quantile(0.25))
smc_df['group_Q3'] = smc_df.groupby(GROUP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.quantile(0.75))

# (2) 수술별 interquartile_range 및 Upper Bound 계산
smc_df['group_IQR'] = smc_df['group_Q3'] - smc_df['group_Q1']
smc_df['group_upper_bound'] = smc_df['group_Q3'] + 1.5 * smc_df['group_IQR']

# (확인용) 수술별 임계값이 잘 들어갔는지 상위 5개만 출력
print(smc_df[[GROUP_COLUMN, TARGET_COLUMN, 'group_upper_bound', MSBOS_COLUMN]].head())

# -------------------------------------------------------------
# 2. 이상치 필터링 조건 생성 (Logic은 유지)
# -------------------------------------------------------------

# 조건 A: 해당 수술의 interquartile_range 상한을 초과하는가?
# (전체 기준이 아니라, 내 수술 그룹의 상한선과 비교)
exceeds_iqr_limit = (smc_df[TARGET_COLUMN] > smc_df['group_upper_bound'])

# 조건 B: MSBOS 값의 2배를 초과하는가? (MSBOS는 이미 수술별 값이므로 그대로 유지)
exceeds_msbos_limit = (smc_df[TARGET_COLUMN] > 2 * smc_df[MSBOS_COLUMN])

# 최종 이상치 조건: A와 B를 동시에 만족할 때만 제거
outlier_mask = np.logical_and(exceeds_iqr_limit, exceeds_msbos_limit)

# -------------------------------------------------------------
# 3. 이상치 제거 및 정리
# -------------------------------------------------------------

# 이상치 개수 확인
outlier_count = outlier_mask.sum()

# 이상치 제거 (필터링)
smc_filtered_df = smc_df[~outlier_mask].copy()

# 계산을 위해 만들었던 임시 컬럼 삭제 (깔끔하게 정리)
smc_filtered_df.drop(columns=['group_Q1', 'group_Q3', 'group_IQR', 'group_upper_bound'], inplace=True, errors='ignore')

# 결과 출력
print("-" * 50)
print(f"[INFO] 제거 전략: [{GROUP_COLUMN}] 그룹별 IQR 적용")
print(f"[INFO] 제거된 이상치 데이터 개수: {outlier_count} 개")
print(f"[INFO] 원래 데이터셋 크기: {len(smc_df)} 행")
print(f"[INFO] 이상치 제거 후 데이터셋 크기: {len(smc_filtered_df)} 행")
print(f"[INFO] 데이터 감소율: {(outlier_count / len(smc_df) * 100):.2f}%")
print("-" * 50)


In [ ]:
MIN_PROCEDURE_COUNT = 50
value_counts = smc_filtered_df['procedure_cd_fin'].value_counts()
frequent_codes = value_counts[value_counts >= MIN_PROCEDURE_COUNT].index
smc_filtered_df = smc_filtered_df[smc_filtered_df['procedure_cd_fin'].isin(frequent_codes)]


In [ ]:
# scikit-learn의 train_test_split 함수를 사용하며, 층화를 위해 'stratify' 옵션을 사용합니다.
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------
# 1. 데이터 준비 (surgery9가 pandas DataFrame으로 로드되어 있다고 가정)
# -------------------------------------------------------------
# TARGET_COLUMN과 DROP_COLUMNS 설정은 이전 코드와 동일하게 유지
RANDOM_STATE = 2025


# 특성(X)과 종속 변수(y) 분리
smc_features_df = smc_filtered_df[feature_columns]
smc_target = smc_filtered_df[TARGET_COLUMN]

# 층화 기준 변수 설정
# One-Hot Encoding 등 전처리는 분할 후에 적용하는 것이 일반적입니다.
procedure_strata = smc_filtered_df['procedure_cd_fin']

# -------------------------------------------------------------
# 2. 층화 무작위 분할 수행 (70% Train, 30% Test)
# -------------------------------------------------------------

# stratify=procedure_strata 옵션을 사용하여 procedure_cd_fin의 비율을 Train/Test에 동일하게 유지합니다.
smc_train_features_df, smc_test_features_df, smc_train_target, smc_test_target = train_test_split(
    smc_features_df,
    smc_target,
    test_size=0.3, # 30%를 테스트 셋으로 사용
    random_state=RANDOM_STATE, # 무작위성 고정
    stratify=procedure_strata # procedure_cd_fin을 층화 기준으로 사용
)

# -------------------------------------------------------------
# 3. 결과 확인
# -------------------------------------------------------------
print("[INFO] Python의 train_test_split을 사용하여 층화 무작위 분할이 완료되었습니다.")
print(f"[INFO] Train Data (특성) 크기: {smc_train_features_df.shape}")
print(f"[INFO] Test Data (특성) 크기: {smc_test_features_df.shape}")

# (선택 사항) 층화 확인: procedure_cd_fin의 비율 비교
print("\n======= [INFO] procedure_cd_fin 비율 비교 (Top 3) =======")
train_ratios = smc_train_features_df['procedure_cd_fin'].value_counts(normalize=True).head(3)
test_ratios = smc_test_features_df['procedure_cd_fin'].value_counts(normalize=True).head(3)
split_ratios_df = pd.DataFrame({'Train Ratio': train_ratios, 'Test Ratio': test_ratios})
print(split_ratios_df)


In [ ]:
# 2. 데이터 전처리 (제한된 컬럼에 맞춰 수정)
# ----------------------------------------------------------------------
def preprocess_data(df):

    # 이진 (0/1) 플래그 컬럼 처리
    binary_flag_cols = [
        'mi', 'pvd', 'mld', 'diabwc', 'rend', 'canc',
        'heparins', 'coumarin_deriatives'
    ]

    for col in binary_flag_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype(int)

    # 범주형 변수
    cat_features = ['surgeon_dept_cd', 'anesthesia_cd', 'procedure_cd_fin']
    for col in cat_features:
        if col in df.columns:
            df[col] = df[col].astype(str)

    target_procedures = ['5059'] # 평균 사용량 10이상인 수술 코드
    if 'procedure_cd_fin' in df.columns:
        for code in target_procedures:
            df[f'procedure_cd_fin_{code}'] = (df['procedure_cd_fin'] == code).astype(int)
        df = df.drop(columns = ['procedure_cd_fin'])
    df = pd.get_dummies(df, columns = ['surgeon_dept_cd', 'anesthesia_cd'])

    # 수치형/임상 수치 컬럼 NaN 처리
    numeric_features_for_imputation = ['age', 'Hb', 'PT_inr', 'Plt', 'aPTT']

    for col in numeric_features_for_imputation:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].mean())

    return df

smc_train_features_df = preprocess_data(smc_train_features_df)
smc_test_features_df = preprocess_data(smc_test_features_df)


In [ ]:
from sklearn.preprocessing import StandardScaler
smc_scaler = StandardScaler()


smc_train_features_df, smc_test_features_df = smc_train_features_df.align(smc_test_features_df, join='outer', axis = 1, fill_value = 0)

numeric_columns = ['age', 'Hb', 'PT_inr', 'Plt', 'aPTT']
scaled_columns = [c for c in numeric_columns if c in smc_train_features_df.columns]

if scaled_columns:
    smc_scaler = StandardScaler()

    smc_train_features_df[scaled_columns] = smc_scaler.fit_transform(smc_train_features_df[scaled_columns])
    smc_test_features_df[scaled_columns] = smc_scaler.transform(smc_test_features_df[scaled_columns])


In [ ]:
smc_train_features_df = smc_train_features_df.astype(np.float32)
smc_test_features_df = smc_test_features_df.astype(np.float32)


In [ ]:
# diabwc 제거, 국소 마취 제거(5)
smc_train_features_df = smc_train_features_df[['Hb', 'PT_inr', 'Plt', 'aPTT', 'anesthesia_cd_1',
       'anesthesia_cd_9', 'canc', 'heparins', 'msbos', 'pvd',
       'surgeon_dept_cd_ENT', 'surgeon_dept_cd_GS', 'surgeon_dept_cd_NS',
       'surgeon_dept_cd_OBGY', 'surgeon_dept_cd_OS', 'surgeon_dept_cd_PS',
       'surgeon_dept_cd_TS', 'surgeon_dept_cd_URO']]
smc_test_features_df = smc_test_features_df[['Hb', 'PT_inr', 'Plt', 'aPTT', 'anesthesia_cd_1',
       'anesthesia_cd_9', 'canc', 'heparins', 'msbos', 'pvd',
       'surgeon_dept_cd_ENT', 'surgeon_dept_cd_GS', 'surgeon_dept_cd_NS',
       'surgeon_dept_cd_OBGY', 'surgeon_dept_cd_OS', 'surgeon_dept_cd_PS',
       'surgeon_dept_cd_TS', 'surgeon_dept_cd_URO']]

feature_count = len(list(smc_train_features_df.columns))


In [ ]:
def adjusted_r2(y_true, y_pred, feature_count):
    """특성 개수를 고려한 Adjusted R-squared를 계산합니다."""
    r2 = r2_score(y_true, y_pred)
    n_samples = len(y_true)

    if n_samples > feature_count + 1:
        adj_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - feature_count - 1)
    else:
        adj_r2 = np.nan

    return adj_r2

def calculate_bootstrap_ci(model, X_test, y_test, n_bootstrap, confidence_level, feature_count):
    random_seed = 2025
    np.random.seed(random_seed)

    lower_p = (100 - confidence_level) / 2
    upper_p = 100 - lower_p

    mae_scores, rmse_scores, r2_scores, adj_r2_scores = [], [], [], []
    n_test = len(X_test)

    print(f"[INFO] [부트스트랩] {n_bootstrap}회 실행 중...")

    for _ in tqdm(range(n_bootstrap)):
        sample_indices = np.random.choice(n_test, size=n_test, replace=True)
        X_sample = X_test.iloc[sample_indices]
        y_sample = y_test.iloc[sample_indices]

        y_pred = model.predict(X_sample)
        y_pred = np.maximum(y_pred, 0)

        mae_scores.append(mean_absolute_error(y_sample, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_sample, y_pred)))
        r2_scores.append(r2_score(y_sample, y_pred))
        adj_r2_scores.append(adjusted_r2(y_sample, y_pred, feature_count))

    results = {
        'Model': [model.__class__.__name__],
        'Mean_RMSE': [np.mean(rmse_scores)], 'RMSE_CI_Lower': [scoreatpercentile(rmse_scores, lower_p)], 'RMSE_CI_Upper': [scoreatpercentile(rmse_scores, upper_p)],
        'Mean_MAE': [np.mean(mae_scores)], 'MAE_CI_Lower': [scoreatpercentile(mae_scores, lower_p)], 'MAE_CI_Upper': [scoreatpercentile(mae_scores, upper_p)],
        'Mean_R_Squared': [np.mean(r2_scores)], 'R2_CI_Lower': [scoreatpercentile(r2_scores, lower_p)], 'R2_CI_Upper': [scoreatpercentile(r2_scores, upper_p)],
        'Mean_Adj_R2': [np.nanmean(adj_r2_scores)],
        'Adj_R2_CI_Lower': [scoreatpercentile(np.array(adj_r2_scores)[~np.isnan(adj_r2_scores)], lower_p)],
        'Adj_R2_CI_Upper': [scoreatpercentile(np.array(adj_r2_scores)[~np.isnan(adj_r2_scores)], upper_p)],
    }

    return pd.DataFrame(results).round(4)


In [ ]:
# ----------------------------------------------------------------------
# 4. 모델 정의 및 확장된 튜닝 설정
# ----------------------------------------------------------------------
cv_strategy = RepeatedKFold(n_splits=10, n_repeats=10, random_state=2025)
scoring_metric = 'neg_root_mean_squared_error'

models = {
    # 1. 다중 선형 회귀 (튜닝 불필요)
    'Linear Regression': (LinearRegression(), {}),

    # 2. 랜덤 포레스트 (RF) - 확장된 튜닝 범위
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), {
        'n_estimators': [100, 300, 500],
        'max_depth': [ 10, 20, 30],
        'min_samples_leaf': [1, 2, 4],
        'min_samples_split': [2, 5, 10]
    }),

    # 3. XGBoost - 확장된 튜닝 범위
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), {
        'n_estimators': [100, 500, 1000], 'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7, 9], 'subsample': [0.7, 1.0], 'colsample_bytree': [0.7, 1.0]
    }),

    # 4. SVR - 확장된 튜닝 범위
    'SVR': (SVR(kernel='rbf'), {
        'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto', 0.1]
    }),

    # 5. ANN (MLP) - 확장된 튜닝 범위
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), {
        'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)],
        'activation': ['relu', 'tanh'],
        'alpha': [0.0001, 0.001, 0.01], 'learning_rate_init': [0.001, 0.01]
    })
}


In [ ]:
#### SMC 모델
smc_best_parameter_records = []
smc_performance_df = pd.DataFrame()
predictions_df = pd.DataFrame({'Actual_Value': smc_test_target})

# ----------------------------------------------------------------------
# 5. 모델 학습, 예측 및 평가 실행 (훈련 성능 및 실행 시간 측정)
# ----------------------------------------------------------------------
print("\n======= [INFO] 모델 학습 및 확장 튜닝 시작 =======\n")

for name, (model, param_grid) in models.items():
    print(f"[INFO] >>> {name} 학습 중...")

    start_time = time.time() # 시간 측정 시작

    # 튜닝 및 학습
    if name == 'Linear Regression':
        trained_model = model.fit(smc_train_features_df, smc_train_target)
        best_params = 'Default'
    else:
        # 튜닝 조합 수 계산 및 진행 정보 출력
        param_counts = [len(v) for v in param_grid.values()]
        total_combinations = np.prod(param_counts)

        # --- CV 정보 고정 및 계산 (10-Fold, 10-Repeats 가정) ---
        n_splits_per_repeat = 10
        n_repeats = 10
        n_splits_total = n_splits_per_repeat * n_repeats # 100회

        # 총 Fit 횟수 계산 (총 조합 수 * 총 폴드 수)
        total_fits = total_combinations * n_splits_total

        print(f"\n[INFO] [CV 전략] {n_splits_per_repeat} K-Fold를 {n_repeats}회 반복 (총 {n_splits_total} 폴드)")
        print(f"[INFO] [Grid Search] 총 {total_combinations}개 조합 테스트 시작")
        print(f"[INFO] (전체 Fit 횟수: {total_fits}회)")

        grid_search = GridSearchCV(model, param_grid, cv=cv_strategy,
                                   scoring=scoring_metric, n_jobs=-1,verbose= 3)
        grid_search.fit(smc_train_features_df, smc_train_target)

        trained_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        smc_best_parameter_records.append({
            'Model':name,
            'Best_Parameters': best_params
        })
        print(f"[INFO] Best Params: {grid_search.best_params_}")

    smc_best_parameter_records.append({
        'model': name,
        'best_parameters': best_params
    })

    # 훈련 데이터셋 성능 계산 및 출력
    y_pred_train = trained_model.predict(smc_train_features_df)
    y_pred_train = np.maximum(y_pred_train, 0)

    train_rmse = np.sqrt(mean_squared_error(smc_train_target, y_pred_train))
    train_mae = mean_absolute_error(smc_train_target, y_pred_train)
    train_r2 = r2_score(smc_train_target, y_pred_train)
    train_adj_r2 = adjusted_r2(smc_train_target, y_pred_train, feature_count)

    print(f"[INFO] [Train Set Performance]")
    print(f"[INFO] RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}, Adj_R2: {train_adj_r2:.4f}")

    # 테스트 데이터셋 부트스트랩 평가
    bootstrap_metrics_df = calculate_bootstrap_ci(trained_model, smc_test_features_df, smc_test_target,
                                                  n_bootstrap=100, confidence_level=95,
                                                  feature_count = feature_count)

    end_time = time.time() # 시간 측정 종료
    execution_time = end_time - start_time

    # 최종 결과 DataFrame 업데이트
    bootstrap_metrics_df['Model'] = name
    bootstrap_metrics_df['Execution_Time (s)'] = round(execution_time, 2)

    smc_performance_df = pd.concat([smc_performance_df, bootstrap_metrics_df], ignore_index=True)

    # 예측 결과 저장을 위한 예측 수행
    y_pred_full = trained_model.predict(smc_test_features_df)
    predictions_df[f'{name}_Prediction'] = y_pred_full

print("\n======= [INFO] 모델 학습 및 부트스트랩 평가 완료 =======\n")


In [ ]:
pd.DataFrame(smc_best_parameter_records).to_csv(str(OUTPUT_DIR / 'smc_best_parmas.csv'), index=False, encoding = 'utf-8-sig')
smc_performance_df.to_csv(str(OUTPUT_DIR / 'smc_best_performance.csv'), index=False, encoding = 'utf-8-sig')


In [ ]:
# 1. 파일 이름 정의
AMC_INPUT_PATH = str(PROCESSED_DIR / 'amc_tidy.csv')

amc_df = pd.read_csv(AMC_INPUT_PATH, encoding= 'utf-8', dtype='str')
amc_df


In [ ]:
amc_df.columns
amc_df = amc_df.rename(columns = {'anesthesia_cd2':'anesthesia_cd',
                          'use_quan_imputed':'use_quan'})


In [ ]:
columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float


In [ ]:
dtype_int = {col: target_int for col in columns_to_int}
amc_df = amc_df.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
amc_df = amc_df.astype(dtype_float)


In [ ]:
# 사용할 컬럼 목록 정의 (20개)
MODEL_COLUMNS = [
    'age', 'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_COLUMN = 'use_quan'

amc_df = amc_df[MODEL_COLUMNS]
feature_columns = [c for c in MODEL_COLUMNS if c != TARGET_COLUMN]

print("\n======= [INFO] 데이터 로드 및 전처리 완료 =======\n")


In [ ]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 0. 설정
# -------------------------------------------------------------
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'
GROUP_COLUMN = 'procedure_cd_fin' # 그룹핑 기준 변수

amc_df = amc_df.copy()
# -------------------------------------------------------------
# 1. 수술별(procedure_cd_fin) interquartile_range 및 임계값 계산
# -------------------------------------------------------------
print("[INFO] 수술별 그룹핑 및 IQR 계산 중...")

# (1) 각 수술 코드별로 lower_quartile, upper_quartile 계산하여 원본 데이터프레임에 임시 컬럼으로 붙임 (.transform 사용)
# transform은 그룹별로 계산한 값을 원래 행의 위치에 맵핑해줍니다.
amc_df['group_Q1'] = amc_df.groupby(GROUP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.quantile(0.25))
amc_df['group_Q3'] = amc_df.groupby(GROUP_COLUMN)[TARGET_COLUMN].transform(lambda x: x.quantile(0.75))

# (2) 수술별 interquartile_range 및 Upper Bound 계산
amc_df['group_IQR'] = amc_df['group_Q3'] - amc_df['group_Q1']
amc_df['group_upper_bound'] = amc_df['group_Q3'] + 1.5 * amc_df['group_IQR']

# (확인용) 수술별 임계값이 잘 들어갔는지 상위 5개만 출력
print(amc_df[[GROUP_COLUMN, TARGET_COLUMN, 'group_upper_bound', MSBOS_COLUMN]].head())

# -------------------------------------------------------------
# 2. 이상치 필터링 조건 생성 (Logic은 유지)
# -------------------------------------------------------------

# 조건 A: 해당 수술의 interquartile_range 상한을 초과하는가?
# (전체 기준이 아니라, 내 수술 그룹의 상한선과 비교)
exceeds_iqr_limit = (amc_df[TARGET_COLUMN] > amc_df['group_upper_bound'])

# 조건 B: MSBOS 값의 2배를 초과하는가? (MSBOS는 이미 수술별 값이므로 그대로 유지)
exceeds_msbos_limit = (amc_df[TARGET_COLUMN] > 2 * amc_df[MSBOS_COLUMN])

# 최종 이상치 조건: A와 B를 동시에 만족할 때만 제거
outlier_mask = np.logical_and(exceeds_iqr_limit, exceeds_msbos_limit)

# -------------------------------------------------------------
# 3. 이상치 제거 및 정리
# -------------------------------------------------------------

# 이상치 개수 확인
outlier_count = outlier_mask.sum()

# 이상치 제거 (필터링)
amc_filtered_df = amc_df[~outlier_mask].copy()

# 계산을 위해 만들었던 임시 컬럼 삭제 (깔끔하게 정리)
amc_filtered_df.drop(columns=['group_Q1', 'group_Q3', 'group_IQR', 'group_upper_bound'], inplace=True, errors='ignore')

# 결과 출력
print("-" * 50)
print(f"[INFO] 제거 전략: [{GROUP_COLUMN}] 그룹별 IQR 적용")
print(f"[INFO] 제거된 이상치 데이터 개수: {outlier_count} 개")
print(f"[INFO] 원래 데이터셋 크기: {len(amc_df)} 행")
print(f"[INFO] 이상치 제거 후 데이터셋 크기: {len(amc_filtered_df)} 행")
print(f"[INFO] 데이터 감소율: {(outlier_count / len(amc_df) * 100):.2f}%")
print("-" * 50)


In [ ]:
MIN_PROCEDURE_COUNT = 50
value_counts = amc_filtered_df['procedure_cd_fin'].value_counts()
frequent_codes = value_counts[value_counts >= MIN_PROCEDURE_COUNT].index
amc_filtered_df = amc_filtered_df[amc_filtered_df['procedure_cd_fin'].isin(frequent_codes)]


In [ ]:
# scikit-learn의 train_test_split 함수를 사용하며, 층화를 위해 'stratify' 옵션을 사용합니다.
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------
# 1. 데이터 준비 (surgery9가 pandas DataFrame으로 로드되어 있다고 가정)
# -------------------------------------------------------------
# TARGET_COLUMN과 DROP_COLUMNS 설정은 이전 코드와 동일하게 유지
RANDOM_STATE = 2025


# 특성(X)과 종속 변수(y) 분리
amc_features_df = amc_filtered_df[feature_columns]
amc_target = amc_filtered_df[TARGET_COLUMN]

# 층화 기준 변수 설정
# One-Hot Encoding 등 전처리는 분할 후에 적용하는 것이 일반적입니다.
procedure_strata = amc_filtered_df['procedure_cd_fin']

# -------------------------------------------------------------
# 2. 층화 무작위 분할 수행 (70% Train, 30% Test)
# -------------------------------------------------------------

# stratify=procedure_strata 옵션을 사용하여 procedure_cd_fin의 비율을 Train/Test에 동일하게 유지합니다.
amc_train_features_df, amc_test_features_df, amc_train_target, amc_test_target = train_test_split(
    amc_features_df,
    amc_target,
    test_size=0.3, # 30%를 테스트 셋으로 사용
    random_state=RANDOM_STATE, # 무작위성 고정
    stratify=procedure_strata # procedure_cd_fin을 층화 기준으로 사용
)

# -------------------------------------------------------------
# 3. 결과 확인
# -------------------------------------------------------------
print("[INFO] Python의 train_test_split을 사용하여 층화 무작위 분할이 완료되었습니다.")
print(f"[INFO] Train Data (특성) 크기: {amc_train_features_df.shape}")
print(f"[INFO] Test Data (특성) 크기: {amc_test_features_df.shape}")

# (선택 사항) 층화 확인: procedure_cd_fin의 비율 비교
print("\n======= [INFO] procedure_cd_fin 비율 비교 (Top 3) =======")
train_ratios = amc_train_features_df['procedure_cd_fin'].value_counts(normalize=True).head(3)
test_ratios = amc_test_features_df['procedure_cd_fin'].value_counts(normalize=True).head(3)
split_ratios_df = pd.DataFrame({'Train Ratio': train_ratios, 'Test Ratio': test_ratios})
print(split_ratios_df)


In [ ]:
# 2. 데이터 전처리 (제한된 컬럼에 맞춰 수정)
# ----------------------------------------------------------------------
def preprocess_data(df):

    # 이진 (0/1) 플래그 컬럼 처리
    binary_flag_cols = [
        'mi', 'pvd', 'mld', 'diabwc', 'rend', 'canc',
        'heparins', 'coumarin_deriatives'
    ]

    for col in binary_flag_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype(int)

    # 범주형 변수
    cat_features = ['surgeon_dept_cd', 'anesthesia_cd', 'procedure_cd_fin']
    for col in cat_features:
        if col in df.columns:
            df[col] = df[col].astype(str)

    target_procedures = ['5059'] # 평균 사용량 10이상인 수술 코드
    if 'procedure_cd_fin' in df.columns:
        for code in target_procedures:
            df[f'procedure_cd_fin_{code}'] = (df['procedure_cd_fin'] == code).astype(int)
        df = df.drop(columns = ['procedure_cd_fin'])
    df = pd.get_dummies(df, columns = ['surgeon_dept_cd', 'anesthesia_cd'])

    # 수치형/임상 수치 컬럼 NaN 처리
    numeric_features_for_imputation = ['age', 'Hb', 'PT_inr', 'Plt', 'aPTT']

    for col in numeric_features_for_imputation:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].mean())

    return df

amc_train_features_df = preprocess_data(amc_train_features_df)
amc_test_features_df = preprocess_data(amc_test_features_df)


In [ ]:
from sklearn.preprocessing import StandardScaler
amc_scaler = StandardScaler()


amc_train_features_df, amc_test_features_df = amc_train_features_df.align(amc_test_features_df, join='outer', axis = 1, fill_value = 0)

numeric_columns = ['age', 'Hb', 'PT_inr', 'Plt', 'aPTT']
scaled_columns = [c for c in numeric_columns if c in amc_train_features_df.columns]

if scaled_columns:
    amc_scaler = StandardScaler()

    amc_train_features_df[scaled_columns] = amc_scaler.fit_transform(amc_train_features_df[scaled_columns])
    amc_test_features_df[scaled_columns] = amc_scaler.transform(amc_test_features_df[scaled_columns])


In [ ]:
amc_train_features_df = amc_train_features_df.astype(np.float32)
amc_test_features_df = amc_test_features_df.astype(np.float32)


In [ ]:
# diabwc 제거, 국소 마취 제거(5)
amc_train_features_df = amc_train_features_df[['Hb', 'PT_inr', 'Plt', 'aPTT', 'anesthesia_cd_1',
       'anesthesia_cd_9', 'canc', 'heparins', 'msbos', 'pvd',
       'surgeon_dept_cd_ENT', 'surgeon_dept_cd_GS', 'surgeon_dept_cd_NS',
       'surgeon_dept_cd_OBGY', 'surgeon_dept_cd_OS', 'surgeon_dept_cd_PS',
       'surgeon_dept_cd_TS', 'surgeon_dept_cd_URO']]
amc_test_features_df = amc_test_features_df[['Hb', 'PT_inr', 'Plt', 'aPTT', 'anesthesia_cd_1',
       'anesthesia_cd_9', 'canc', 'heparins', 'msbos', 'pvd',
       'surgeon_dept_cd_ENT', 'surgeon_dept_cd_GS', 'surgeon_dept_cd_NS',
       'surgeon_dept_cd_OBGY', 'surgeon_dept_cd_OS', 'surgeon_dept_cd_PS',
       'surgeon_dept_cd_TS', 'surgeon_dept_cd_URO']]

feature_count = len(list(amc_train_features_df))


In [ ]:
### AMC 모델


amc_best_parameter_records = []
amc_performance_df = pd.DataFrame()
predictions_df = pd.DataFrame({'Actual_Value': amc_test_target})

# ----------------------------------------------------------------------
# 5. 모델 학습, 예측 및 평가 실행 (훈련 성능 및 실행 시간 측정)
# ----------------------------------------------------------------------
print("\n======= [INFO] 모델 학습 및 확장 튜닝 시작 =======\n")

for name, (model, param_grid) in models.items():
    print(f"[INFO] >>> {name} 학습 중...")

    start_time = time.time() # 시간 측정 시작

    # 튜닝 및 학습
    if name == 'Linear Regression':
        trained_model = model.fit(amc_train_features_df, amc_train_target)
        best_params = 'Default'
    else:
        # 튜닝 조합 수 계산 및 진행 정보 출력
        param_counts = [len(v) for v in param_grid.values()]
        total_combinations = np.prod(param_counts)

        # --- CV 정보 고정 및 계산 (10-Fold, 10-Repeats 가정) ---
        n_splits_per_repeat = 10
        n_repeats = 10
        n_splits_total = n_splits_per_repeat * n_repeats # 100회

        # 총 Fit 횟수 계산 (총 조합 수 * 총 폴드 수)
        total_fits = total_combinations * n_splits_total

        print(f"\n[INFO] [CV 전략] {n_splits_per_repeat} K-Fold를 {n_repeats}회 반복 (총 {n_splits_total} 폴드)")
        print(f"[INFO] [Grid Search] 총 {total_combinations}개 조합 테스트 시작")
        print(f"[INFO] (전체 Fit 횟수: {total_fits}회)")

        grid_search = GridSearchCV(model, param_grid, cv=cv_strategy,
                                   scoring=scoring_metric, n_jobs=-1,verbose= 3)
        grid_search.fit(amc_train_features_df, amc_train_target)

        trained_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        amc_best_parameter_records.append({
            'Model':name,
            'Best_Parameters': best_params
        })
        print(f"[INFO] Best Params: {grid_search.best_params_}")



    # 훈련 데이터셋 성능 계산 및 출력
    y_pred_train = trained_model.predict(amc_train_features_df)
    y_pred_train = np.maximum(y_pred_train, 0)

    train_rmse = np.sqrt(mean_squared_error(amc_train_target, y_pred_train))
    train_mae = mean_absolute_error(amc_train_target, y_pred_train)
    train_r2 = r2_score(amc_train_target, y_pred_train)
    train_adj_r2 = adjusted_r2(amc_train_target, y_pred_train, feature_count)

    print(f"[INFO] [Train Set Performance]")
    print(f"[INFO] RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}, Adj_R2: {train_adj_r2:.4f}")

    # 테스트 데이터셋 부트스트랩 평가
    bootstrap_metrics_df = calculate_bootstrap_ci(trained_model, amc_test_features_df, amc_test_target,
                                                  n_bootstrap=100, confidence_level=95,
                                                  feature_count=feature_count)

    end_time = time.time() # 시간 측정 종료
    execution_time = end_time - start_time

    # 최종 결과 DataFrame 업데이트
    bootstrap_metrics_df['Model'] = name
    bootstrap_metrics_df['Execution_Time (s)'] = round(execution_time, 2)

    amc_performance_df = pd.concat([amc_performance_df, bootstrap_metrics_df], ignore_index=True)

    # 예측 결과 저장을 위한 예측 수행
    y_pred_full = trained_model.predict(amc_test_features_df)
    predictions_df[f'{name}_Prediction'] = y_pred_full

print("\n======= [INFO] 모델 학습 및 부트스트랩 평가 완료 =======\n")


In [ ]:
pd.DataFrame(amc_best_parameter_records).to_csv(str(OUTPUT_DIR / 'best_params_amc.csv'), index=0, encoding = 'utf-8-sig')
amc_performance_df.to_csv(str(OUTPUT_DIR / 'performance_amc.csv'), index=0, encoding = 'utf-8-sig')


In [ ]:
# 통합 데이터
pooled_train_features_df = pd.concat([smc_train_features_df, amc_train_features_df]).reset_index(drop=True)
pooled_test_features_df = pd.concat([smc_test_features_df, amc_test_features_df]).reset_index(drop=True)
pooled_train_target = pd.concat([smc_train_target, amc_train_target]).reset_index(drop=True)
pooled_test_target = pd.concat([smc_test_target, amc_test_target]).reset_index(drop=True)


In [ ]:
feature_count = len(list(pooled_train_features_df))


In [ ]:
# 통합 모델

pooled_best_parameter_records = []
pooled_performance_df = pd.DataFrame()
predictions_df = pd.DataFrame({'Actual_Value': pooled_test_target})

# ----------------------------------------------------------------------
# 5. 모델 학습, 예측 및 평가 실행 (훈련 성능 및 실행 시간 측정)
# ----------------------------------------------------------------------
print("\n======= [INFO] 모델 학습 및 확장 튜닝 시작 =======\n")

for name, (model, param_grid) in models.items():
    print(f"[INFO] >>> {name} 학습 중...")

    start_time = time.time() # 시간 측정 시작

    # 튜닝 및 학습
    if name == 'Linear Regression':
        trained_model = model.fit(pooled_train_features_df, pooled_train_target)
        best_params = 'Default'
    else:
        # 튜닝 조합 수 계산 및 진행 정보 출력
        param_counts = [len(v) for v in param_grid.values()]
        total_combinations = np.prod(param_counts)

        # --- CV 정보 고정 및 계산 (10-Fold, 10-Repeats 가정) ---
        n_splits_per_repeat = 10
        n_repeats = 10
        n_splits_total = n_splits_per_repeat * n_repeats # 100회

        # 총 Fit 횟수 계산 (총 조합 수 * 총 폴드 수)
        total_fits = total_combinations * n_splits_total

        print(f"\n[INFO] [CV 전략] {n_splits_per_repeat} K-Fold를 {n_repeats}회 반복 (총 {n_splits_total} 폴드)")
        print(f"[INFO] [Grid Search] 총 {total_combinations}개 조합 테스트 시작")
        print(f"[INFO] (전체 Fit 횟수: {total_fits}회)")

        grid_search = GridSearchCV(model, param_grid, cv=cv_strategy,
                                   scoring=scoring_metric, n_jobs=-1,verbose= 3)
        grid_search.fit(pooled_train_features_df, pooled_train_target)

        trained_model = grid_search.best_estimator_
        best_params = grid_search.best_params_

        pooled_best_parameter_records.append({
            'Model':name,
            'Best_Parameters': best_params
        })
        print(f"[INFO] Best Params: {grid_search.best_params_}")



    # 훈련 데이터셋 성능 계산 및 출력
    y_pred_train = trained_model.predict(pooled_train_features_df)
    y_pred_train = np.maximum(y_pred_train, 0)

    train_rmse = np.sqrt(mean_squared_error(pooled_train_target, y_pred_train))
    train_mae = mean_absolute_error(pooled_train_target, y_pred_train)
    train_r2 = r2_score(pooled_train_target, y_pred_train)
    train_adj_r2 = adjusted_r2(pooled_train_target, y_pred_train, feature_count)

    print(f"[INFO] [Train Set Performance]")
    print(f"[INFO] RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}, Adj_R2: {train_adj_r2:.4f}")

    # 테스트 데이터셋 부트스트랩 평가
    bootstrap_metrics_df = calculate_bootstrap_ci(trained_model, pooled_test_features_df, pooled_test_target,
                                                  n_bootstrap=100, confidence_level=95,
                                                  feature_count=feature_count)

    end_time = time.time() # 시간 측정 종료
    execution_time = end_time - start_time

    # 최종 결과 DataFrame 업데이트
    bootstrap_metrics_df['Model'] = name
    bootstrap_metrics_df['Execution_Time (s)'] = round(execution_time, 2)

    pooled_performance_df = pd.concat([pooled_performance_df, bootstrap_metrics_df], ignore_index=True)

    # 예측 결과 저장을 위한 예측 수행
    y_pred_full = trained_model.predict(pooled_test_features_df)
    predictions_df[f'{name}_Prediction'] = y_pred_full

print("\n======= [INFO] 모델 학습 및 부트스트랩 평가 완료 =======\n")


In [ ]:
pd.DataFrame(pooled_best_parameter_records).to_csv(str(OUTPUT_DIR / 'best_params_total.csv'), index=0, encoding = 'utf-8-sig')
pooled_performance_df.to_csv(str(OUTPUT_DIR / 'performance_total.csv'), index=0, encoding = 'utf-8-sig')


In [ ]:
# smc_df 모델 amc_df 성능

# 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요!
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 500, 'subsample': 0.7},
    'SVR': {'C': 10, 'gamma': 'auto'},
    'ANN (MLP)': {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
cross_site_performance_df = pd.DataFrame()

# 전제 변수 (이전 단계에서 정의된 값을 사용해야 함)
# X_train, y_train, feature_count 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("\n======= [INFO] 최적 모델 기반 훈련 데이터셋 CI 계산 시작 =======")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n[INFO] >>> {name} 재학습 및 훈련 CI 계산 중...")

    start_time = time.time()

    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()

    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025

        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(smc_train_features_df, smc_train_target)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    evaluation_metrics_df = calculate_bootstrap_ci(
        trained_model,
        amc_test_features_df,
        amc_test_target,
        n_bootstrap=100,
        confidence_level=95,
        feature_count=feature_count
    )

    end_time = time.time()
    execution_time = end_time - start_time

    # 결과 저장
    evaluation_metrics_df['Model'] = name
    evaluation_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    cross_site_performance_df = pd.concat([cross_site_performance_df, evaluation_metrics_df], ignore_index=True)

print("\n======= [INFO] 훈련 데이터셋 CI 계산 완료 =======\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("\n======= [INFO] 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) =======")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
cross_site_performance_df = cross_site_performance_df[output_cols]
print(cross_site_performance_df)#.to_markdown(index=False))


In [ ]:
# smc_df 모델 total test

# 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요!
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 500, 'subsample': 0.7},
    'SVR': {'C': 10, 'gamma': 'auto'},
    'ANN (MLP)': {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
cross_site_performance_df = pd.DataFrame()

# 전제 변수 (이전 단계에서 정의된 값을 사용해야 함)
# X_train, y_train, feature_count 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("\n======= [INFO] 최적 모델 기반 훈련 데이터셋 CI 계산 시작 =======")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n[INFO] >>> {name} 재학습 및 훈련 CI 계산 중...")

    start_time = time.time()

    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()

    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025

        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(smc_train_features_df, smc_train_target)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    evaluation_metrics_df = calculate_bootstrap_ci(
        trained_model,
        pooled_test_features_df,
        pooled_test_target,
        n_bootstrap=100,
        confidence_level=95,
        feature_count=feature_count
    )

    end_time = time.time()
    execution_time = end_time - start_time

    # 결과 저장
    evaluation_metrics_df['Model'] = name
    evaluation_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    cross_site_performance_df = pd.concat([cross_site_performance_df, evaluation_metrics_df], ignore_index=True)

print("\n======= [INFO] 훈련 데이터셋 CI 계산 완료 =======\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("\n======= [INFO] 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) =======")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
cross_site_performance_df = cross_site_performance_df[output_cols]
print(cross_site_performance_df)#.to_markdown(index=False))


In [ ]:
### amc_df 모델 smc_df 테스트

# 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요!
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.7},
    'SVR': {'C': 10, 'gamma': 'scale'},
    'ANN (MLP)': {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
cross_site_performance_df = pd.DataFrame()

# 전제 변수 (이전 단계에서 정의된 값을 사용해야 함)
# X_train, y_train, feature_count 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("\n======= [INFO] 최적 모델 기반 훈련 데이터셋 CI 계산 시작 =======")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n[INFO] >>> {name} 재학습 및 훈련 CI 계산 중...")

    start_time = time.time()

    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()

    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025

        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(amc_train_features_df, amc_train_target)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    evaluation_metrics_df = calculate_bootstrap_ci(
        trained_model,
        smc_test_features_df,
        smc_test_target,
        n_bootstrap=100,
        confidence_level=95,
        feature_count=feature_count
    )

    end_time = time.time()
    execution_time = end_time - start_time

    # 결과 저장
    evaluation_metrics_df['Model'] = name
    evaluation_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    cross_site_performance_df = pd.concat([cross_site_performance_df, evaluation_metrics_df], ignore_index=True)

print("\n======= [INFO] 훈련 데이터셋 CI 계산 완료 =======\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("\n======= [INFO] 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) =======")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
cross_site_performance_df = cross_site_performance_df[output_cols]
print(cross_site_performance_df)#.to_markdown(index=False))


In [ ]:
### amc_df 모델 total 테스트

# 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요!
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.7},
    'SVR': {'C': 10, 'gamma': 'scale'},
    'ANN (MLP)': {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
cross_site_performance_df = pd.DataFrame()

# 전제 변수 (이전 단계에서 정의된 값을 사용해야 함)
# X_train, y_train, feature_count 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("\n======= [INFO] 최적 모델 기반 훈련 데이터셋 CI 계산 시작 =======")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n[INFO] >>> {name} 재학습 및 훈련 CI 계산 중...")

    start_time = time.time()

    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()

    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025

        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(amc_train_features_df, amc_train_target)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    evaluation_metrics_df = calculate_bootstrap_ci(
        trained_model,
        pooled_test_features_df,
        pooled_test_target,
        n_bootstrap=100,
        confidence_level=95,
        feature_count=feature_count
    )

    end_time = time.time()
    execution_time = end_time - start_time

    # 결과 저장
    evaluation_metrics_df['Model'] = name
    evaluation_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    cross_site_performance_df = pd.concat([cross_site_performance_df, evaluation_metrics_df], ignore_index=True)

print("\n======= [INFO] 훈련 데이터셋 CI 계산 완료 =======\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("\n======= [INFO] 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) =======")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
cross_site_performance_df = cross_site_performance_df[output_cols]
print(cross_site_performance_df)#.to_markdown(index=False))


In [ ]:
# 통합 모델 smc_df 성능

# 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요!
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 1000, 'subsample': 1.0},
    'SVR': {'C': 10, 'gamma': 'auto'},
    'ANN (MLP)': {'activation': 'tanh', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
cross_site_performance_df = pd.DataFrame()

# 전제 변수 (이전 단계에서 정의된 값을 사용해야 함)
# X_train, y_train, feature_count 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("\n======= [INFO] 최적 모델 기반 훈련 데이터셋 CI 계산 시작 =======")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n[INFO] >>> {name} 재학습 및 훈련 CI 계산 중...")

    start_time = time.time()

    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()

    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025

        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(pooled_train_features_df, pooled_train_target)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    evaluation_metrics_df = calculate_bootstrap_ci(
        trained_model,
        smc_test_features_df,
        smc_test_target,
        n_bootstrap=100,
        confidence_level=95,
        feature_count=feature_count
    )

    end_time = time.time()
    execution_time = end_time - start_time

    # 결과 저장
    evaluation_metrics_df['Model'] = name
    evaluation_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    cross_site_performance_df = pd.concat([cross_site_performance_df, evaluation_metrics_df], ignore_index=True)

print("\n======= [INFO] 훈련 데이터셋 CI 계산 완료 =======\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("\n======= [INFO] 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) =======")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
cross_site_performance_df = cross_site_performance_df[output_cols]
print(cross_site_performance_df)#.to_markdown(index=False))


In [ ]:
# 통합 모델 amc_df 성능

# 여기에 이전 Grid Search에서 얻은 최적 하이퍼파라미터 값을 입력하세요!
BEST_PARAMS = {
    'Random Forest': {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 500},
    'XGBoost':{'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 1000, 'subsample': 1.0},
    'SVR': {'C': 10, 'gamma': 'auto'},
    'ANN (MLP)': {'activation': 'tanh', 'alpha': 0.01, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001}
}

# ----------------------------------------------------------------------
# 2. 모델 정의 및 학습 데이터 CI 계산 (최적 하이퍼파라미터 적용)
# ----------------------------------------------------------------------
import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# (adjusted_r2 및 calculate_bootstrap_ci 함수는 이미 정의되어 있다고 가정)

# 성능 결과를 저장할 DataFrame
cross_site_performance_df = pd.DataFrame()

# 전제 변수 (이전 단계에서 정의된 값을 사용해야 함)
# X_train, y_train, feature_count 는 이미 정의된 상태여야 합니다.

models_to_evaluate = {
    'Linear Regression': (LinearRegression(), {}),
    'Random Forest': (RandomForestRegressor(random_state=2025, n_jobs=-1), BEST_PARAMS['Random Forest']),
    'XGBoost': (XGBRegressor(random_state=2025, n_jobs=-1, objective='reg:squarederror'), BEST_PARAMS['XGBoost']),
    'SVR': (SVR(kernel='rbf'), BEST_PARAMS['SVR']),
    'ANN (MLP)': (MLPRegressor(random_state=2025, max_iter=500, activation='relu'), BEST_PARAMS['ANN (MLP)'])
}

print("\n======= [INFO] 최적 모델 기반 훈련 데이터셋 CI 계산 시작 =======")

for name, (base_model, params) in models_to_evaluate.items():
    print(f"\n[INFO] >>> {name} 재학습 및 훈련 CI 계산 중...")

    start_time = time.time()

    # 1. 최적 하이퍼파라미터를 적용하여 모델 인스턴스 생성
    model_params = params.copy()

    if name == 'Linear Regression':
        trained_model = base_model
    else:
        # randomstate 필요 없는 모델 제외
        if 'random_state' in base_model.__init__.__code__.co_varnames:
            model_params['random_state'] = 2025

        trained_model = base_model.__class__(**model_params)

    # 2. 훈련 데이터로 모델 학습
    trained_model.fit(pooled_train_features_df, pooled_train_target)

    # 3. 훈련 데이터셋에 대한 부트스트랩 CI 계산
    evaluation_metrics_df = calculate_bootstrap_ci(
        trained_model,
        amc_test_features_df,
        amc_test_target,
        n_bootstrap=100,
        confidence_level=95,
        feature_count=feature_count
    )

    end_time = time.time()
    execution_time = end_time - start_time

    # 결과 저장
    evaluation_metrics_df['Model'] = name
    evaluation_metrics_df['Execution_Time (s)'] = round(execution_time, 2)
    cross_site_performance_df = pd.concat([cross_site_performance_df, evaluation_metrics_df], ignore_index=True)

print("\n======= [INFO] 훈련 데이터셋 CI 계산 완료 =======\n")

# ----------------------------------------------------------------------
# 3. 최종 결과 출력
# ----------------------------------------------------------------------
print("="*120)
print("\n======= [INFO] 훈련 데이터셋 최종 성능 비교 (95% 부트스트랩 신뢰구간) =======")
print("="*120)

# 출력 순서 재정렬
output_cols = [
    'Model', 'Execution_Time (s)',
    'Mean_RMSE', 'RMSE_CI_Lower', 'RMSE_CI_Upper',
    'Mean_MAE', 'MAE_CI_Lower', 'MAE_CI_Upper',
    'Mean_Adj_R2', 'Adj_R2_CI_Lower', 'Adj_R2_CI_Upper'
]

# RMSE의 평균(Mean_RMSE) 기준으로 정렬
cross_site_performance_df = cross_site_performance_df[output_cols]
print(cross_site_performance_df)#.to_markdown(index=False))
